# 沪深 300 指数涨跌方向的机器学习分类预测

**——基于 10 日滚动均值构造的 A1~A7 特征，对比 RBF-SVM、逻辑回归、随机森林、AdaBoost 四类模型**

| 项目 | 说明 |
|---|---|
| 数据文件 | `index300.xlsx`（国泰安 CSMAR 指数日行情库 · 沪深 300 指数） |
| 样本区间 | 2014-01-02 ~ 2014-12-31，共 245 个交易日 |


**7 个特征的定义（窗口均为 10 个交易日，公式中 $MA_{10}(\cdot)$ 表示 10 日滚动均值，含当日）**

| 特征 | 计算公式 | 刻画的内容 |
|---|---|---|
| A1 | $\text{收盘}_t \,/\, MA_{10}(\text{收盘})_t$ | 收盘价在 10 日均线上方还是下方（趋势位置、超买超卖） |
| A2 | $\text{成交量}_t \,/\, MA_{10}(\text{成交量})_t$ | 量能的相对强弱 |
| A3 | 当日日回报率（Idxtrd08，%） | 动量 |
| A4 | $\text{最高}_t \,/\, MA_{10}(\text{收盘})_t$ | 当日上冲力度 |
| A5 | $\text{最低}_t \,/\, MA_{10}(\text{收盘})_t$ | 当日下档支撑水平 |
| A6 | $\text{最高}_t - \text{最低}_t$ | 日内振幅（波动幅度） |
| A7 | $\text{收盘}_t - \text{开盘}_t$ | K 线实体，区分收阳 / 收阴 |


说明：A3（日回报率）和 A6、A7（点位差）本身不需要滚动均值；需要 10 日均线的是 A1、A2、A4、A5。题目所说"10 期滚动均值会产生前 9 行缺失值"正是来自这 4 个特征的 `rolling(10)`。

## 1. 数据预处理

**原始字段含义**（CSMAR 指数日行情库）：

| 字段 | 含义 | 
|---|---|
| `Indexcd` | 指数代码（000300 = 沪深 300） | 
| `Idxtrd01` | 交易日期 | 
| `Idxtrd02` | 开盘指数 | 
| `Idxtrd03` | 最高指数 | 
| `Idxtrd04` | 最低指数 | 
| `Idxtrd05` | 收盘指数 | 
| `Idxtrd06` | 成份证券成交量 | 
| `Idxtrd07` | 成份证券成交额 | 
| `Idxtrd08` | 指数日回报率（%） | 

先读取数据、统一列名、按日期排序，并查看规模与取值范围。

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rcParams
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             confusion_matrix, precision_score, recall_score,
                             f1_score)

warnings.filterwarnings("ignore")
rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "SimSun", "DejaVu Sans"]
rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110
rcParams["savefig.dpi"] = 150
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.unicode.east_asian_width", True)

SEED = 42                                  # 全局随机种子，保证可复现
WINDOW = 10                                # 滚动窗口：10 个交易日
N_TEST = 30                                # 测试集样本数（题目规定）
FIG_DIR = "figs"                           # 图片输出目录
os.makedirs(FIG_DIR, exist_ok=True)

DATA_PATH = r"index300.xlsx"
def save_show(fig, filename):
    fig.savefig(os.path.join(FIG_DIR, filename), bbox_inches="tight")
    plt.show()

raw = pd.read_excel(DATA_PATH, sheet_name="Sheet1")
print("原始数据维度：", raw.shape)
RENAME = {
    "Indexcd":  "index_cd",
    "Idxtrd01": "date",
    "Idxtrd02": "open",
    "Idxtrd03": "high",
    "Idxtrd04": "low",
    "Idxtrd05": "close",
    "Idxtrd06": "volume",
    "Idxtrd07": "amount",
    "Idxtrd08": "ret",
}
df = raw.rename(columns=RENAME).copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)   # 严格按时间升序

display(df.head(5).round(4))

原始数据维度： (245, 9)


,index_cd,date,open,high,low,close,volume,amount,ret
0,300,2014-01-02,2323.43,2325.99,2310.65,2321.98,451942.91,4901221.11,-0.3454
1,300,2014-01-03,2311.97,2314.84,2280.89,2290.78,597826.45,5773970.99,-1.3436
2,300,2014-01-06,2286.37,2286.37,2229.33,2238.64,663004.03,5997936.01,-2.2762
3,300,2014-01-07,2222.31,2246.79,2218.65,2238.00,437531.03,4256564.81,-0.0284
4,300,2014-01-08,2240.64,2262.58,2228.42,2241.91,513488.54,5069148.89,0.1747


**数据检查**：简单检查一下缺失值、日期重复/乱序。

In [2]:
print("各列缺失值个数：")
print(df.isna().sum().to_string())

print("\n重复日期个数：", int(df["date"].duplicated().sum()))
print("日期范围：", df["date"].min().date(), "→", df["date"].max().date(),
      "，共", len(df), "个交易日")
print("日期是否严格递增：", bool(df["date"].is_monotonic_increasing))

各列缺失值个数：
index_cd    0
date        0
open        0
high        0
low         0
close       0
volume      0
amount      0
ret         0

重复日期个数： 0
日期范围： 2014-01-02 → 2014-12-31 ，共 245 个交易日
日期是否严格递增： True


**检查结论**：245 行、无缺失值、无重复日期、日期严格递增。**数据质量良好，无需插补或剔除异常值**，可以直接进入特征构造。

## 2. 特征工程：构造 A1~A7

这里**先在全样本上计算滚动均值、再统一剔除缺失行**：滚动统计量本身只依赖历史，不会产生未来信息泄漏；若先切分再算，反而会让训练集末尾的样本丢掉 9 天历史。

In [121]:
# 收盘价的 10 日均线、成交量的 10 日均线
ma_close = df["close"].rolling(WINDOW).mean()      
ma_volume = df["volume"].rolling(WINDOW).mean()

features = pd.DataFrame({
    "A1": df["close"] / ma_close,                  # 收盘相对 10 日均线位置
    "A2": df["volume"] / ma_volume,                # 量能相对强弱
    "A3": df["ret"],                               # 当日日回报率（%），动量
    "A4": df["high"] / ma_close,                   # 上冲力度
    "A5": df["low"] / ma_close,                    # 下档支撑
    "A6": df["high"] - df["low"],                  # 日内振幅
    "A7": df["close"] - df["open"],                # K 线实体
})

display(pd.DataFrame({
    "date": df["date"].dt.date,
    "close": df["close"],
    "MA10_close": ma_close.round(2),
    "A1": features["A1"].round(4),
    "A2": features["A2"].round(4),
}).head(12))

print("\nA1~A7 描述性统计")
display(features.describe().T.round(4))

,date,close,MA10_close,A1,A2
0,2014-01-02,2321.98,NaN,NaN,NaN
1,2014-01-03,2290.78,NaN,NaN,NaN
2,2014-01-06,2238.64,NaN,NaN,NaN
3,2014-01-07,2238.00,NaN,NaN,NaN
4,2014-01-08,2241.91,NaN,NaN,NaN
5,2014-01-09,2222.22,NaN,NaN,NaN
6,2014-01-10,2204.85,NaN,NaN,NaN
7,2014-01-13,2193.68,NaN,NaN,NaN
8,2014-01-14,2212.85,NaN,NaN,NaN
9,2014-01-15,2208.94,2237.38,0.9873,0.9244



A1~A7 描述性统计


,count,mean,std,min,25%,50%,75%,max
A1,236.0,1.0084,0.0233,0.9509,0.9935,1.0048,1.0185,1.1153
A2,236.0,1.0437,0.2564,0.5712,0.8627,0.9909,1.1980,2.0131
A3,245.0,0.1774,1.2129,-4.4873,-0.4801,0.0550,0.7832,4.6099
A4,236.0,1.0150,0.0246,0.9593,0.9994,1.0092,1.0250,1.1451
A5,236.0,0.9994,0.0187,0.9471,0.9868,0.9984,1.0083,1.0608
A6,245.0,38.4756,34.0477,9.7300,20.5000,29.1600,40.0600,313.3100
A7,245.0,5.0709,30.0939,-126.1900,-9.7400,2.0200,17.5100,144.6100


为了解决显著的量纲差异对依赖于欧氏距离计算的模型（RBF 核支持向量机、逻辑回归等）的直接影响，后面统一在 `Pipeline` 中加入 `StandardScaler`（**只用训练集拟合均值方差**）。标准化对树模型无影响，为保持流程一致也一并放入。

## 3. 标签构造：二分类

题目规定：**若后一期收盘指数大于当期收盘指数，标记为上涨（标签 1）；否则标记为下跌或持平（标签 - 1）**。

$$y_t=\begin{cases}1, & \text{close}_{t+1} > \text{close}_t \\ -1, & \text{close}_{t+1} \le \text{close}_t\end{cases}$$

细节：
**最后一个交易日没有"下期"**：`close.shift(-1)` 的最后一项是 `NaN`。**如果直接写 `np.where(next_close > close, 1, -1)`，`NaN > x` 会得到 `False`，末行会被错误地标成 -1**。为解决，此处首先将末行标签置为缺失，再统一剔除。

In [122]:
next_close = df["close"].shift(-1)                     # 下一交易日收盘价
label = pd.Series(np.where(next_close > df["close"], 1, -1),
                  index=df.index, dtype="float64")
label[next_close.isna()] = np.nan                      # 末日置为缺失

print("标签分布（构造后、剔除缺失前）：")
print(pd.Series({
    "上涨 (1)": int((label == 1).sum()),
    "下跌或持平 (-1) ": int((label == -1).sum()),
}).to_string())

print("\n最后 7 个交易日的标签情况：")
display(pd.DataFrame({
    "date": df["date"].dt.date.tail(7),
    "close": df["close"].tail(7),
    "next_close": next_close.tail(7),
    "label": label.tail(7),
}))

标签分布（构造后、剔除缺失前）：
上涨 (1)            130
下跌或持平 (-1)     114

最后 7 个交易日的标签情况：


,date,close,next_close,label
238,2014-12-23,3324.92,3230.39,-1.0
239,2014-12-24,3230.39,3335.42,1.0
240,2014-12-25,3335.42,3445.84,1.0
241,2014-12-26,3445.84,3455.46,1.0
242,2014-12-29,3455.46,3457.55,1.0
243,2014-12-30,3457.55,3533.71,1.0
244,2014-12-31,3533.71,NaN,NaN


## 4. 数据对齐与划分

**训练集**：前 205 个样本（2014-01-15 ~ 2014-11-18），**测试集**：最后 30 个样本（2014-11-19 ~ 2014-12-30）

In [123]:
FEATURES = [f"A{i}" for i in range(1, 8)]
data = pd.concat([
    df[["date", "open", "high", "low", "close", "volume"]],
    features,
    label.rename("y"),
], axis=1)

n_raw = len(data)
data = data.dropna().reset_index(drop=True)     # 一次性剔除前 9 行滚动缺失和末行标签缺失
data["y"] = data["y"].astype(int)

print(f"最终可用样本日期范围: {data['date'].min().date()} → {data['date'].max().date()}")

display(data.head(5).round(4))

最终可用样本日期范围: 2014-01-15 → 2014-12-30


,date,open,high,low,close,volume,A1,A2,A3,A4,A5,A6,A7,y
0,2014-01-15,2210.02,2215.90,2193.80,2208.94,489623.95,0.9873,0.9244,-0.1765,0.9904,0.9805,22.10,-1.08,1
1,2014-01-16,2209.45,2227.26,2201.56,2211.84,522989.02,0.9935,0.9743,0.1314,1.0004,0.9889,25.70,2.39,-1
2,2014-01-17,2203.82,2205.97,2175.88,2178.49,502731.55,0.9835,0.9535,-1.5081,0.9959,0.9823,30.09,-25.33,-1
3,2014-01-20,2173.65,2182.46,2156.46,2165.99,405601.54,0.9810,0.8087,-0.5736,0.9885,0.9767,26.00,-7.66,1
4,2014-01-21,2168.42,2197.92,2168.42,2187.41,403062.47,0.9930,0.8092,0.9888,0.9978,0.9844,29.50,18.99,1


In [147]:
# 按时间顺序切分
n = len(data)
train_df = data.iloc[: n - N_TEST].copy()      # 前 205 个
test_df = data.iloc[n - N_TEST:].copy()        # 后 30 个
X_train = train_df[FEATURES].to_numpy(dtype=float)
y_train = train_df["y"].to_numpy()
X_test = test_df[FEATURES].to_numpy(dtype=float)
y_test = test_df["y"].to_numpy()

def describe_labels(name, y):
    up = int((y == 1).sum())
    dn = int((y == -1).sum())
    print(f"{name}: n={len(y):3d} | 上涨 {up:3d} 个（{up / len(y):.2%}）| 下跌 {dn:3d} 个（{dn / len(y):.2%}）")

print(f"训练集：{train_df['date'].min().date()} → {train_df['date'].max().date()}")
describe_labels("", y_train)
print(f"测试集：{test_df['date'].min().date()} → {test_df['date'].max().date()}")
describe_labels("", y_test)

训练集：2014-01-15 → 2014-11-18
: n=205 | 上涨 104 个（50.73%）| 下跌 101 个（49.27%）
测试集：2014-11-19 → 2014-12-30
: n= 30 | 上涨  24 个（80.00%）| 下跌   6 个（20.00%）


## 5. 比较四种分类模型

**建模约定**：

1. **标准化** `Pipeline([("scaler", StandardScaler()), ("clf", 模型)])`：
`StandardScaler` 的均值/标准差**只在训练集上拟合**。
1. **固定 `random_state=42`**。
2. **评价指标**：主指标是题目要求的**准确率**（训练集、测试集各一个）。

In [179]:
MODELS = {
    "RBF-SVM":   SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED),
    "逻辑回归":   LogisticRegression(max_iter=1000, random_state=SEED),
    "随机森林":   RandomForestClassifier(n_estimators=100, random_state=SEED),
    "AdaBoost":  AdaBoostClassifier(n_estimators=50, random_state=SEED),
}
results = []
fitted = {}         

for name, clf in MODELS.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", clf)])
    pipe.fit(X_train, y_train)                       # 只在训练集上 fit

    pred_train = pipe.predict(X_train)
    pred_test = pipe.predict(X_test)

    results.append({
        "模型": name,
        "训练集准确率": accuracy_score(y_train, pred_train),
        "测试集准确率": accuracy_score(y_test, pred_test),
        "测试集预测「上涨」个数": int((pred_test == 1).sum()),
        "测试集预测「下跌」个数": int((pred_test == -1).sum()),
    })
    fitted[name] = pipe

res_df = pd.DataFrame(results)

print("四种模型的训练集/测试集准确率")
display(res_df[["模型", "训练集准确率", "测试集准确率",
                "测试集预测「上涨」个数", "测试集预测「下跌」个数"]].round(4))
print("训练集准确率排名：", " > ".join(res_df.sort_values("训练集准确率", ascending=False)["模型"]))
print("测试集准确率排名：", " > ".join(res_df.sort_values("测试集准确率", ascending=False)["模型"]))

四种模型的训练集/测试集准确率


,模型,训练集准确率,测试集准确率,测试集预测「上涨」个数,测试集预测「下跌」个数
0,RBF-SVM,0.6390,0.6333,23,7
1,逻辑回归,0.6098,0.5333,16,14
2,随机森林,1.0000,0.4333,13,17
3,AdaBoost,0.7268,0.6667,24,6


训练集准确率排名： 随机森林 > AdaBoost > RBF-SVM > 逻辑回归
测试集准确率排名： AdaBoost > RBF-SVM > 逻辑回归 > 随机森林
